# Model Router with a System One Model

This notebook runs the [model-router](./README.md) harness: an agent loop where [CLM-8B](../../inference/system-one/clm-8b/) routes each task to a model tier, gates risky tool calls and escalates answers that do not answer the request. It runs the same tasks with both tier configurations, then measures the router on a larger task suite:

- [tiers/hybrid.yaml](./tiers/hybrid.yaml): Qwen3-8B on your EKS GPUs, served by Dynamo, with Claude on Amazon Bedrock for harder tasks.
- [tiers/bedrock.yaml](./tiers/bedrock.yaml): Claude Haiku, Sonnet and Opus on Amazon Bedrock.

Before you start, deploy the dependencies in [Deploy the dependencies](./README.md#deploy-the-dependencies): CLM-8B, and for the hybrid configuration, Qwen3-8B with tool calling.

## Setup and Imports

In [ ]:
! pip install -r requirements.txt

In [ ]:
import json
import os
import subprocess
import sys

# Set working directory to this example
os.chdir(os.path.expanduser('~/amazon-eks-machine-learning-with-terraform-and-kubeflow/examples/agentic/model-router'))
sys.path.insert(0, os.getcwd())

namespace = 'kubeflow-user-example-com'
os.environ.setdefault('CLM_BASE_URL', 'http://localhost:8700')
os.environ.setdefault('DYNAMO_BASE_URL', 'http://localhost:8000/v1')
# os.environ['AWS_REGION'] = 'aws-region'   # the Region the harness calls Bedrock in, if not set already

## Step 1: Check the Dependencies

The harness runs here and reaches the services through port-forwards. Start them in a terminal and leave them running:

    kubectl port-forward -n kubeflow-user-example-com svc/clm-serve 8700:8700
    kubectl port-forward -n kubeflow-user-example-com svc/dyn-qwen3-8b-frontend 8000:8000   # hybrid only

First, CLM-8B. `/health` returns 200 even while the encoder is unreachable, so check the `embedder` field.

In [ ]:
from src.clm import CLM

clm = CLM(os.environ['CLM_BASE_URL'], api_key=os.environ.get('CLM_API_KEY'))
assert clm.healthy(), 'clm-serve is not ready: check the port-forward and that /health shows "embedder": true'
print('clm-serve is ready')

Next, Qwen3-8B on Dynamo, for the hybrid configuration. [tool_call_check.py](../../inference/dynamo/vllm/qwen3-8b/tool_call_check.py) checks that tool calls come back as structured `tool_calls`. It fails if `dyn-qwen3-8b` was launched from `dgd-agg.yaml` instead of `dgd-agg-tools.yaml`.

In [ ]:
cmd = [sys.executable, '../../inference/dynamo/vllm/qwen3-8b/tool_call_check.py',
       '--base-url', os.environ['DYNAMO_BASE_URL']]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout, result.stderr)
assert result.returncode == 0, 'tool calling is not working on the Dynamo deployment'

Last, Amazon Bedrock. One short Converse call to each Bedrock model in the two tiers files confirms access.

In [ ]:
from src.backends import make_backend
from src.loop import load_tiers

configs = {name: load_tiers(f'tiers/{name}.yaml') for name in ['hybrid', 'bedrock']}
bedrock_tiers = {t['model']: t for cfg in configs.values() for t in cfg['tiers'] if t['backend'] == 'bedrock'}
for model, tier in bedrock_tiers.items():
    reply = make_backend(tier).chat('Reply with one word.', [{'role': 'user', 'content': 'Say ready.'}], [], 16)
    print(f'{model}: {reply.text.strip()!r}')

## Step 2: Route the Tasks

[tasks/demo.jsonl](./tasks/demo.jsonl) has ten tasks: lookups and arithmetic that need a tool, short writing, refunds and an email that act on a customer's behalf, and two hard technical questions. CLM-8B answers the `signals` questions in each tiers file about every task, and the policy picks the first tier whose `profile.max` limits the answers stay within. No model is called yet.

In [ ]:
from src import hooks, policy

tasks = [json.loads(line) for line in open('tasks/demo.jsonl') if line.strip()]

for name, cfg in configs.items():
    print(f'--- {name}')
    for t in tasks:
        s = hooks.signals(clm, t['task'], cfg['signals'])
        i, reasons = policy.choose(cfg['tiers'], s.value, cfg['policy'], cfg['policy'])
        print(f"{t['id']:20} {cfg['tiers'][i]['name']:8} {'; '.join(reasons) or 'within the first tier limits'}")

## Step 3: Run the Hybrid Configuration

Each task runs through the agent loop on the tier it was routed to. For every task you see the route, the tools each tier called (with the gate's verdict on `issue_refund` and `send_email`), any escalation, and the final answer. The full traces, including token usage and timings, go to `results/hybrid.jsonl`.

In [ ]:
from run import show
from src.loop import run_task

def run_all(cfg, tasks, start_tier=None):
    os.makedirs('results', exist_ok=True)
    traces = []
    with open(f"results/{cfg['name']}.jsonl", 'a') as f:
        for t in tasks:
            trace = run_task(t['task'], cfg, clm, start_tier=start_tier)
            show(t['id'], trace)
            f.write(json.dumps({'id': t['id'], **trace.to_dict()}) + '\n')
            traces.append(trace)
    return traces

hybrid_traces = run_all(configs['hybrid'], tasks)

## Step 4: Run the Bedrock Configuration

The same tasks, with every tier on Amazon Bedrock. The traces go to `results/bedrock.jsonl`.

In [ ]:
bedrock_traces = run_all(configs['bedrock'], tasks)

## Step 5: See the Gate in Front of the Small Model

Larger models often refuse a risky request by themselves, so the gate has the most to do in front of a small model. Here the three tasks with risky tools start on the hybrid configuration's small tier, Qwen3-8B, instead of being routed. When the gate blocks a call, the model receives `blocked: this action needs the user's confirmation` as the tool result.

In [ ]:
risky = [t for t in tasks if t['id'] in {'refund-requested', 'refund-unrequested', 'email'}]
gate_traces = run_all(configs['hybrid'], risky, start_tier='small')

## Step 6: Check the Runs

These checks are about the harness, not the quality of the answers: every task ended with an answer from some tier, every call to a risky tool went through the gate, and every escalation moved up exactly one tier.

In [ ]:
from src.tools import TOOLS

def check(traces):
    for tr in traces:
        assert tr.answer, f'no final answer for: {tr.task}'
        names = [a.tier_name for a in tr.attempts]
        for e, (lower, upper) in zip(tr.escalations, zip(names, names[1:])):
            assert (e['from'], e['to']) == (lower, upper), f'escalation out of order for: {tr.task}'
        for a in tr.attempts:
            for c in a.tool_calls:
                if TOOLS.get(c['name']) and TOOLS[c['name']].risky:
                    assert 'gate' in c, f"{c['name']} ran without the gate for: {tr.task}"

for label, traces in [('hybrid', hybrid_traces), ('bedrock', bedrock_traces), ('gate', gate_traces)]:
    check(traces)
    gated = sum('gate' in c for tr in traces for a in tr.attempts for c in a.tool_calls)
    escalated = sum(bool(tr.escalations) for tr in traces)
    print(f'{label}: {len(traces)} tasks answered, {gated} gated tool calls, {escalated} escalated tasks')

## Step 7: Evaluate the Router

The demo tasks show what each hook does; they are too few to judge the routing. [tasks/eval.jsonl](./tasks/eval.jsonl) has 42 tasks, each with a check in code that decides whether an answer passed. `eval.py` runs every task on every tier, then compares always using one tier, the router, the router with escalation, and an oracle that knows the first tier that passes. It also shows how well each signal predicts that the small tier fails.

This step calls every tier on every task, so it takes a while. The runs and the measured latency and tokens go to `results/`. See [Evaluate the router](./README.md#evaluate-the-router) for how to read the report and fit the limits.

In [ ]:
for name in ['hybrid', 'bedrock']:
    result = subprocess.run([sys.executable, 'eval.py', '--tiers', f'tiers/{name}.yaml'], capture_output=True, text=True)
    assert result.returncode == 0, result.stderr
    print(f'--- {name}')
    print(result.stdout.split('rows written to')[-1])

## Step 8: Clean Up

Stop the port-forwards. The harness itself runs nothing in the cluster. To remove the dependencies, follow the *Stop* sections of the [CLM-8B](../../inference/system-one/clm-8b/README.md#stop-the-services) and [Qwen3-8B](../../inference/dynamo/vllm/qwen3-8b/README.md#stop-service) examples.